In [1]:
# Parameters
base_path = "C:\\Users\\ander\\OneDrive\\TCC_USP"
run_id = "20251117_120549"


In [2]:
# 1. Setup de caminhos locais
import os
import subprocess
from datetime import datetime
from pathlib import Path

from src.io import paths
from src.config import loader as cfg

DATA_PATHS = paths.get_data_paths()
PROJECT_PATHS = paths.get_project_paths()
BASE_PATH = str(DATA_PATHS["base"])
RAW_PATH = str(DATA_PATHS["data_raw"])
PROC_PATH = str(DATA_PATHS["data_processed"])
INTERIM_PATH = str(DATA_PATHS["data_interim"])
CONFIG = cfg.load_config()

NB_NAME = "02_baseline_logit"
STAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

print("BASE_PATH:", BASE_PATH)
print("PROC_PATH:", PROC_PATH)


BASE_PATH: C:\Users\ander\OneDrive\TCC_USP
PROC_PATH: C:\Users\ander\OneDrive\TCC_USP\data_processed


In [3]:
# 2. Carregar dados processados
import pandas as pd, numpy as np
from sklearn.model_selection import TimeSeriesSplit
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score

df_ibov = pd.read_csv(os.path.join(PROC_PATH, "ibovespa_clean.csv"))
df_news = pd.read_csv(os.path.join(PROC_PATH, "noticias_clean.csv"))

# corrigir nomes das colunas
df_ibov["data"] = pd.to_datetime(df_ibov["data"])
df_news["data"] = pd.to_datetime(df_news["data"])

In [4]:
# 3. Feature simples: sentimento proxy por comprimento (placeholder)
df_news["sent_len"] = df_news["clean_text"].fillna("").apply(lambda s: len(s.split()))
daily = df_news.groupby("data")["sent_len"].mean().reset_index()

In [5]:
# 4. Merge com IBOV e rÃ³tulo (direÃ§Ã£o D+1)
df = pd.merge(df_ibov.sort_values("data"), daily, on="data", how="left").fillna(0)
df["ret1"] = df["close"].pct_change().shift(-1)           # retorno do dia seguinte
df = df.dropna().copy()
df["y"] = (df["ret1"] > 0).astype(int)
X = df[["sent_len"]].values
y = df["y"].values

In [6]:
# 5. ValidaÃ§Ã£o temporal (walk-forward simples)
tscv = TimeSeriesSplit(n_splits=3)
aucs, mdas = [], []
for tr, te in tscv.split(X):
    clf = LogisticRegression(max_iter=2000).fit(X[tr], y[tr])
    proba = clf.predict_proba(X[te])[:,1]
    pred  = (proba > 0.5).astype(int)
    aucs.append(roc_auc_score(y[te], proba))
    mdas.append(accuracy_score(y[te], pred))

print({
    "1d": {"AUC": float(np.mean(aucs)), "MDA": float(np.mean(mdas))}
})

{'1d': {'AUC': 0.6111111111111112, 'MDA': 0.75}}
